# 04 - Estudio de Ablación y Evaluación de Modelos (Bake-Off)

En este cuaderno evaluaremos 6 algoritmos de clasificación distintos frente al reto de predecir 84 clases simultáneas.

Los contendientes seleccionados son:
1. **Multinomial Naive Bayes:** Algoritmo estadístico clásico, muy rápido para textos.
2. **Linear SVC:** Máquinas de Vectores de Soporte, ideal para matrices de alta dimensionalidad.
3. **Regresión Logística:** El baseline matemático estándar.
4. **Random Forest:** Algoritmo basado en árboles paralelos (Bagging).
5. **LightGBM:** Algoritmo avanzado de Gradient Boosting (crecimiento por hojas).
6. **XGBoost:** El estándar de la industria en Gradient Boosting (crecimiento por niveles).

Seguiremos un diseño experimental estricto: primero los evaluaremos con los datos crudos ("Sin SMOTE") para ver cómo se comportan ante el desbalanceo. Posteriormente, aplicaremos balanceo sintético para medir la mejora de las métricas.

In [1]:
import pandas as pd
from scipy import sparse

# Importamos las 6 familias de algoritmos
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

print("--- FASE 1: CARGA DE DATOS Y PREPARACIÓN DE MODELOS ---")

# 1. Cargamos las matrices numéricas
X_train = sparse.load_npz("../data/features/en_X_train_tfidf.npz")
X_test = sparse.load_npz("../data/features/en_X_test_tfidf.npz")

# 2. Cargamos las etiquetas (La tripleta)
y_train = pd.read_csv("../data/features/en_y_train.csv")['target']
y_test = pd.read_csv("../data/features/en_y_test.csv")['target']

print(f"Datos originales cargados. Tickets de entrenamiento: {X_train.shape[0]}")

# 3. Empaquetamos los 6 modelos en un diccionario para poder iterar sobre ellos
# Usamos n_jobs=-1 para que usen todos los núcleos de tu procesador
# Usamos random_state=42 para que el TFM sea reproducible
modelos = {
    "Naive Bayes": MultinomialNB(),
    "Linear SVC": LinearSVC(random_state=42, max_iter=2000),
    "Regresion Logistica": LogisticRegression(random_state=42, max_iter=2000),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    
    # Añadimos verbose=-1 para silenciar los warnings de LightGBM
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    
    # Añadimos verbosity=0 para asegurar que XGBoost también se calle
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss', verbosity=0)
}

print("Los 6 contendientes están inicializados y listos en la memoria RAM.")

--- FASE 1: CARGA DE DATOS Y PREPARACIÓN DE MODELOS ---
Datos originales cargados. Tickets de entrenamiento: 18493
Los 6 contendientes están inicializados y listos en la memoria RAM.


### Paso 2: Bucle de Entrenamiento (Línea Base Sin SMOTE)

En esta fase entrenaremos los 6 algoritmos usando la matriz original desbalanceada. 

**Consideraciones técnicas:**
* Utilizaremos `LabelEncoder` para transformar las clases categóricas (texto) a números enteros, ya que las librerías modernas de Boosting (XGBoost, LightGBM) no aceptan *strings* como variable objetivo.
* Monitorizaremos especialmente la métrica **F1-Macro**, ya que el F1-Micro puede darnos una falsa sensación de precisión al estar dominado por las clases mayoritarias. El F1-Macro hace la media no ponderada, castigando a los algoritmos que ignoran las clases raras.

In [2]:
import time
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder
from IPython.display import display

# --- 1. Conversión de Etiquetas (Label Encoding) ---
# Transformamos las etiquetas de texto a números enteros (0 a 83) para XGBoost/LightGBM
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# --- 2. Bucle de Entrenamiento ---
print("--- FASE 2: ENTRENAMIENTO SIN BALANCEAR (SIN SMOTE) ---\n")

# Lista vacía donde iremos guardando el registro de cada modelo
resultados_sin_smote = []

for nombre, modelo in modelos.items():
    print(f"⏳ Entrenando {nombre}...")
    
    # Arrancamos el cronómetro
    start_time = time.time()
    
    # Entrenamos el modelo con los datos crudos
    modelo.fit(X_train, y_train_encoded)
    
    # Medimos el tiempo
    tiempo_entrenamiento = round(time.time() - start_time, 2)
    
    # Hacemos el examen final (predicciones sobre el conjunto Test)
    y_pred = modelo.predict(X_test)
    
    # Calculamos todas las métricas
    # zero_division=0 evita errores si un modelo nunca predice una clase rara
    f1_macro = round(f1_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    f1_micro = round(f1_score(y_test_encoded, y_pred, average='micro', zero_division=0), 4)
    precision = round(precision_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    recall = round(recall_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    
    # Guardamos los resultados de este modelo
    resultados_sin_smote.append({
        'Modelo': nombre,
        'Condicion': 'Sin SMOTE',
        'F1-Macro': f1_macro,
        'F1-Micro': f1_micro,
        'Precision (Macro)': precision,
        'Recall (Macro)': recall,
        'Tiempo (seg)': tiempo_entrenamiento
    })
    
    print(f"✅ Completado -> F1-Macro: {f1_macro} | Tiempo: {tiempo_entrenamiento}s")

# Convertimos la lista de diccionarios en un DataFrame de Pandas
df_resultados_sin_smote = pd.DataFrame(resultados_sin_smote)

print("\n--- RESULTADOS GLOBALES (SIN SMOTE) ---")
display(df_resultados_sin_smote)

--- FASE 2: ENTRENAMIENTO SIN BALANCEAR (SIN SMOTE) ---

⏳ Entrenando Naive Bayes...
✅ Completado -> F1-Macro: 0.028 | Tiempo: 0.04s
⏳ Entrenando Linear SVC...
✅ Completado -> F1-Macro: 0.4702 | Tiempo: 11.07s
⏳ Entrenando Regresion Logistica...
✅ Completado -> F1-Macro: 0.1553 | Tiempo: 10.93s
⏳ Entrenando Random Forest...
✅ Completado -> F1-Macro: 0.6538 | Tiempo: 6.26s
⏳ Entrenando LightGBM...
✅ Completado -> F1-Macro: 0.0045 | Tiempo: 92.24s
⏳ Entrenando XGBoost...
✅ Completado -> F1-Macro: 0.4278 | Tiempo: 351.98s

--- RESULTADOS GLOBALES (SIN SMOTE) ---


,Modelo,Condicion,F1-Macro,F1-Micro,Precision (Macro),Recall (Macro),Tiempo (seg)
0,Naive Bayes,Sin SMOTE,0.0280,0.1721,0.0949,0.0428,0.04
1,Linear SVC,Sin SMOTE,0.4702,0.4250,0.5393,0.4343,11.07
2,Regresion Logistica,Sin SMOTE,0.1553,0.2824,0.3000,0.1472,10.93
3,Random Forest,Sin SMOTE,0.6538,0.6064,0.8596,0.5629,6.26
4,LightGBM,Sin SMOTE,0.0045,0.0606,0.0045,0.0100,92.24
5,XGBoost,Sin SMOTE,0.4278,0.4379,0.6047,0.3671,351.98


### Paso 3: Balanceo (SMOTE) y Entrenamiento Final

Tras comprobar que la mayoría de algoritmos fracasan al enfrentarse a las clases raras, y observar problemas de convergencia matemática en algoritmos Boosting debido a la alta dispersión (sparsity), aplicaremos **SMOTE**.

Esta técnica inyectará tickets sintéticos en las clases minoritarias para igualar matemáticamente el conjunto de datos. Una vez balanceado el `Train`, volveremos a entrenar a los 6 algoritmos, evaluaremos sobre el `Test` (que permanece inalterado) y exportaremos el histórico de métricas.

In [3]:
from imblearn.over_sampling import SMOTE

print("--- FASE 3: BALANCEO (SMOTE) Y ENTRENAMIENTO FINAL ---\n")

# 1. Aplicamos SMOTE
print("⏳ Aplicando SMOTE a la matriz de entrenamiento...")
start_smote = time.time()
smote = SMOTE(random_state=42)

# OJO: Se lo aplicamos a y_train_encoded (los números), no a y_train (el texto)
X_train_smote, y_train_smote_encoded = smote.fit_resample(X_train, y_train_encoded)

print(f"✅ SMOTE completado en {round(time.time() - start_smote, 2)}s.")
print(f"Nuevas dimensiones de Train: {X_train_smote.shape[0]} tickets (Todas las clases igualadas)\n")

# Lista para guardar los resultados balanceados
resultados_con_smote = []

# 2. Bucle de entrenamiento con los datos inyectados
for nombre, modelo in modelos.items():
    print(f"⏳ Entrenando {nombre} (Con SMOTE)...")
    start_time = time.time()
    
    # Entrenamos con la nueva matriz GIGANTE balanceada
    modelo.fit(X_train_smote, y_train_smote_encoded)
    tiempo_entrenamiento = round(time.time() - start_time, 2)
    
    # Predecimos sobre el Test (el Test original de siempre)
    y_pred = modelo.predict(X_test)
    
    f1_macro = round(f1_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    f1_micro = round(f1_score(y_test_encoded, y_pred, average='micro', zero_division=0), 4)
    precision = round(precision_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    recall = round(recall_score(y_test_encoded, y_pred, average='macro', zero_division=0), 4)
    
    resultados_con_smote.append({
        'Modelo': nombre,
        'Condicion': 'Con SMOTE',
        'F1-Macro': f1_macro,
        'F1-Micro': f1_micro,
        'Precision (Macro)': precision,
        'Recall (Macro)': recall,
        'Tiempo (seg)': tiempo_entrenamiento
    })
    
    print(f"✅ Completado -> F1-Macro: {f1_macro} | Tiempo: {tiempo_entrenamiento}s")

# 3. Juntamos todo en un DataFrame maestro (Tracker)
df_resultados_con_smote = pd.DataFrame(resultados_con_smote)
df_tracker_final = pd.concat([df_resultados_sin_smote, df_resultados_con_smote], ignore_index=True)

# 4. Guardamos el histórico oficial en disco
df_tracker_final.to_csv("../data/processed/tracker_en.csv", index=False)

print("\n--- TRACKER FINAL EXPORTADO A DISCO ---")
# Lo mostramos bonito, ordenado por modelo para poder comparar cara a cara
display(df_tracker_final.sort_values(by=['Modelo', 'Condicion']))

--- FASE 3: BALANCEO (SMOTE) Y ENTRENAMIENTO FINAL ---

⏳ Aplicando SMOTE a la matriz de entrenamiento...
✅ SMOTE completado en 0.54s.
Nuevas dimensiones de Train: 152376 tickets (Todas las clases igualadas)

⏳ Entrenando Naive Bayes (Con SMOTE)...
✅ Completado -> F1-Macro: 0.3051 | Tiempo: 0.43s
⏳ Entrenando Linear SVC (Con SMOTE)...
✅ Completado -> F1-Macro: 0.4768 | Tiempo: 80.93s
⏳ Entrenando Regresion Logistica (Con SMOTE)...
✅ Completado -> F1-Macro: 0.4201 | Tiempo: 74.17s
⏳ Entrenando Random Forest (Con SMOTE)...
✅ Completado -> F1-Macro: 0.6818 | Tiempo: 131.84s
⏳ Entrenando LightGBM (Con SMOTE)...
✅ Completado -> F1-Macro: 0.0013 | Tiempo: 622.33s
⏳ Entrenando XGBoost (Con SMOTE)...
✅ Completado -> F1-Macro: 0.4386 | Tiempo: 1429.66s

--- TRACKER FINAL EXPORTADO A DISCO ---


,Modelo,Condicion,F1-Macro,F1-Micro,Precision (Macro),Recall (Macro),Tiempo (seg)
10,LightGBM,Con SMOTE,0.0013,0.0134,0.0040,0.0128,622.33
4,LightGBM,Sin SMOTE,0.0045,0.0606,0.0045,0.0100,92.24
7,Linear SVC,Con SMOTE,0.4768,0.4066,0.4695,0.5095,80.93
1,Linear SVC,Sin SMOTE,0.4702,0.4250,0.5393,0.4343,11.07
6,Naive Bayes,Con SMOTE,0.3051,0.2647,0.2974,0.3963,0.43
0,Naive Bayes,Sin SMOTE,0.0280,0.1721,0.0949,0.0428,0.04
9,Random Forest,Con SMOTE,0.6818,0.6499,0.7838,0.6262,131.84
3,Random Forest,Sin SMOTE,0.6538,0.6064,0.8596,0.5629,6.26
8,Regresion Logistica,Con SMOTE,0.4201,0.3430,0.4121,0.4608,74.17
2,Regresion Logistica,Sin SMOTE,0.1553,0.2824,0.3000,0.1472,10.93
